In [1]:
!pip install openai pandas scikit-learn -q

In [2]:
import os
import pandas as pd
from openai import OpenAI

In [3]:
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API')
except Exception:
    OPENROUTER_API_KEY = os.getenv('OPENROUTER_API', 'sk-or-YOUR-KEY-HERE')

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL = "arcee-ai/trinity-large-preview:free"

print(f"Model  : {MODEL}")
print(f"API key: {'SET' if OPENROUTER_API_KEY and 'YOUR-KEY' not in OPENROUTER_API_KEY else 'NOT SET'}")

Model  : arcee-ai/trinity-large-preview:free
API key: SET


In [4]:
df = pd.read_csv("Telco_Customer_Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [17]:

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [6]:
from sklearn.preprocessing import LabelEncoder

df_ml = df.copy()

le = LabelEncoder()

for col in df_ml.columns:
    if df_ml[col].dtype == "object":
        df_ml[col] = le.fit_transform(df_ml[col])

In [7]:
from sklearn.model_selection import train_test_split

X = df_ml.drop("Churn",axis=1)
y = df_ml["Churn"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [8]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=2000)

lr_model.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=2000)

In [9]:
from sklearn.metrics import accuracy_score

y_pred = lr_model.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.8147622427253371


In [10]:
importance_lr = pd.Series(
    abs(lr_model.coef_[0]),
    index=X.columns
).sort_values(ascending=False)

print(importance_lr.head(10))


PhoneService        1.024934
Contract            0.724407
PaperlessBilling    0.347948
OnlineSecurity      0.294523
TechSupport         0.259207
InternetService     0.230785
Dependents          0.189687
SeniorCitizen       0.171182
OnlineBackup        0.157508
DeviceProtection    0.082982
dtype: float64


In [11]:
def build_context(df):

    total = len(df)

    churn_rate = df["Churn"].mean()*100

    ctx = f"""Telco Churn Dataset — {total} customers, {len(df.columns)} columns
Columns: {', '.join(df.columns)}

Churn rate: {churn_rate:.1f}%

Model accuracy: {accuracy:.2f}

Top churn drivers:
{importance_lr.head(5).to_string()}

Churn by Contract:
{df.groupby('Contract')['Churn'].mean().mul(100).round(1).astype(str)+'%'}

Avg Monthly Charges:
Churned: ${df[df['Churn']==1]['MonthlyCharges'].mean():.2f}
Retained: ${df[df['Churn']==0]['MonthlyCharges'].mean():.2f}

Avg Tenure:
Churned: {df[df['Churn']==1]['tenure'].mean():.1f}
Retained: {df[df['Churn']==0]['tenure'].mean():.1f}

Churn by Internet Service:
{df.groupby('InternetService')['Churn'].mean().mul(100).round(1).astype(str)+'%'}

Churn by Payment Method:
{df.groupby('PaymentMethod')['Churn'].mean().mul(100).round(1).astype(str)+'%'}
"""

    return ctx

In [12]:
CONTEXT = build_context(df)

print(CONTEXT)

Telco Churn Dataset — 7043 customers, 21 columns
Columns: customerID, gender, SeniorCitizen, Partner, Dependents, tenure, PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges, Churn

Churn rate: 26.5%

Model accuracy: 0.81

Top churn drivers:
PhoneService        1.024934
Contract            0.724407
PaperlessBilling    0.347948
OnlineSecurity      0.294523
TechSupport         0.259207

Churn by Contract:
Contract
Month-to-month    42.7%
One year          11.3%
Two year           2.8%
Name: Churn, dtype: object

Avg Monthly Charges:
Churned: $74.44
Retained: $61.27

Avg Tenure:
Churned: 18.0
Retained: 37.6

Churn by Internet Service:
InternetService
DSL            19.0%
Fiber optic    41.9%
No              7.4%
Name: Churn, dtype: object

Churn by Payment Method:
PaymentMethod
Bank transfer (automatic)    16.7%
Credit card (automatic

In [13]:
def ask(question):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a data analyst. Answer using only the dataset stats provided."},
            {"role": "user",   "content": f"Dataset:\n{CONTEXT}\n\nQuestion: {question}"}
        ],
        max_tokens=300,
    )

    print(f"Q: {question}")
    print("-" * 50)
    print(response.choices[0].message.content.strip())
    print()

In [14]:
ask("Describe the dataset")




Q: Describe the dataset
--------------------------------------------------
The Telco Churn Dataset contains 7,043 customer records with 21 attributes including demographic information, service subscriptions, contract details, billing information, and churn status. The dataset shows a 26.5% churn rate, meaning approximately 1,867 customers have left the service. The predictive model achieves 81% accuracy in identifying churners.

Key insights reveal that customers with month-to-month contracts are most likely to churn (42.7%), while those with two-year contracts show the lowest churn (2.8%). Fiber optic internet subscribers demonstrate the highest churn rate at 41.9%, compared to 19.0% for DSL users. Payment method analysis shows electronic check users have the highest churn propensity at 45.3%.

Financially, churned customers pay an average of $74.44 monthly, compared to $61.27 for retained customers. Additionally, churned customers have significantly shorter average tenures (18 months

In [15]:
ask("What are the high-risk customer characteristics?")


Q: What are the high-risk customer characteristics?
--------------------------------------------------
Based on the dataset analysis, the high-risk customer characteristics for churn are:

1. Month-to-month contract: 42.7% churn rate
2. Fiber optic internet service: 41.9% churn rate
3. Electronic check payment method: 45.3% churn rate
4. No phone service: Highest impact on churn (1.024934)
5. Paperless billing: 0.347948 impact on churn
6. Lower tenure (avg 18.0 months for churned customers vs 37.6 for retained)
7. Higher monthly charges (avg $74.44 for churned vs $61.27 for retained)
8. Lack of online security: 0.294523 impact on churn
9. Lack of tech support: 0.259207 impact on churn

Customers with these characteristics are most likely to churn and should be prioritized for retention efforts.



In [16]:
ask("Which payment method has the highest churn?")

Q: Which payment method has the highest churn?
--------------------------------------------------
Based on the provided data, Electronic check has the highest churn rate at 45.3%.

